# Step 1 — Get raw video metadata from YouTube

Pipeline stage 1 of the DEI project: for every **CEO / year** pair in `data/ceos.csv`,
search YouTube for `"[CEO] interview"` restricted to that year and store **all** the
metadata of **every** video the API returns.

**Output:** `data/output/videos_metadata/all_videos.json` — one record per CEO/year pair:

```json
{
  "year": 2026, "company": "Amazon", "rank": 1, "CEO": "Andy Jassy",
  "query": "Andy Jassy interview",
  "n_videos_found": 50,
  "videos": [ { ...full YouTube metadata... } ]
}
```

- `n_videos_found` is an **integer** (possibly `0`) when the CEO is known.
- `n_videos_found` is the string **`"NA"`** when the CEO name is missing from `ceos.csv` —
  no search was possible, which is different from "searched and found nothing".

**Resumable:** the collection cell can be re-run as many times as needed. Pairs that
already have a result are skipped, never overwritten. Progress is flushed to disk after
*every* CEO, so an interruption (or exhausted quota) never loses work.

**Size:** roughly 5 KB per video → expect the finished `all_videos.json` to land around
**0.5–0.7 GB**. That is well past GitHub's 100 MB per-file limit, so add
`data/output/` to `.gitignore` (or track it with Git LFS) before committing.

**Quota:** each pair costs 101 units (`search.list` = 100, `videos.list` = 1). A YouTube
API key gets 10,000 units/day → ~99 pairs per key per day. Keys rotate automatically when
one runs out; when all are exhausted the run stops cleanly and you pick it up tomorrow.

In [ ]:
# !pip install google-api-python-client isodate python-dotenv pandas

## 1. Configuration

In [54]:
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# ── Paths ────────────────────────────────────────────────────────────────────
ENV_FILE   = Path(".env")
CEOS_CSV   = Path("data/ceos.csv")
OUTPUT_DIR = Path("data/output/videos_metadata")
OUTPUT_JSON  = OUTPUT_DIR / "all_videos.json"   # the deliverable
OUTPUT_JSONL = OUTPUT_DIR / "all_videos.jsonl"  # append-only checkpoint (crash safety)
KEY_PROJECTS_CACHE = OUTPUT_DIR / "key_projects.json"  # which Cloud project each API key belongs to

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Search parameters ────────────────────────────────────────────────────────
QUERY_TEMPLATE = "{ceo} interview"
MAX_RESULTS    = 50    # YouTube search.list hard cap per request

# ── Re-run behaviour ─────────────────────────────────────────────────────────
# Pairs already collected are always skipped. These two flags control the edge cases:
RETRY_NA   = True   # re-try pairs stored as "NA" if a CEO name has since been filled in
RETRY_ZERO = False  # re-search pairs that legitimately returned 0 videos

# Stop the run after this many searches (None = keep going until quota runs out).
MAX_SEARCHES_THIS_RUN = None

# Optional: restrict this run to specific years, e.g. [2020, 2021]. None = all years.
YEARS_FILTER = None

# ── Quota bookkeeping ────────────────────────────────────────────────────────
COST_SEARCH = 100
COST_VIDEOS = 1

print(f"CEOs file  : {CEOS_CSV}")
print(f"Output JSON: {OUTPUT_JSON}")

CEOs file  : data/ceos.csv
Output JSON: data/output/videos_metadata/all_videos.json


## 2. API key pool with automatic rotation

Loads every `YOUTUBE_API_KEY_*` from `.env` and rotates to the next key when one runs dry.
When every key is exhausted, `AllKeysExhausted` is raised — the collection loop catches it,
saves, and stops cleanly.

**Detecting an exhausted key is the subtle part.** YouTube reports a spent daily quota two
different ways:

| HTTP | `reason` | meaning |
|------|----------|---------|
| 403 | `quotaExceeded` / `dailyLimitExceeded` | key is done for the day |
| 429 | `rateLimitExceeded` + *"…per day…"* in the message | **also** done for the day |
| 429 | `rateLimitExceeded`, no "per day" | genuine burst limit — just slow down |

The `reason` alone is ambiguous, so the **message text** decides. A daily-quota error rotates
the key; a burst limit retries on the same key with exponential backoff.

**Quota is per Google Cloud project, not per key.** Four keys created inside one project all
draw on the same 10,000 units/day, and rotating between them buys nothing. Google reveals the
project number in the quota error (`consumer 'project_number:…'`), so the pool records it in
`key_projects.json`. Once known, exhausting one key immediately retires its project-mates
instead of wasting a call on each, and a warning is printed at startup.

In [55]:
load_dotenv(ENV_FILE)

# YouTube signals an exhausted DAILY quota in two different ways:
#   403 quotaExceeded / dailyLimitExceeded
#   429 rateLimitExceeded with "Quota exceeded ... per day" in the message  ← looks transient, is not
# Only a short-term burst limit is genuinely retryable, so the message text decides.
QUOTA_REASONS     = {"quotaExceeded", "dailyLimitExceeded"}
TRANSIENT_REASONS = {"rateLimitExceeded", "userRateLimitExceeded", "backendError", "internalError"}

# Phrases that mean "this key is done for the day" even under a transient-looking reason.
DAILY_QUOTA_MARKERS = ("per day", "perday", "dailylimit", "quota exceeded for quota metric")


class AllKeysExhausted(RuntimeError):
    """Every configured YouTube API key has hit its daily quota."""


def _error_info(err: HttpError) -> tuple[str, str, str]:
    """Return (reason, message, project_number) parsed out of an HttpError."""
    reason = message = project = ""
    try:
        payload = json.loads(err.content.decode("utf-8"))
        error = payload.get("error", {})
        message = error.get("message", "") or ""
        errors = error.get("errors") or []
        if errors and isinstance(errors[0], dict):
            reason = errors[0].get("reason", "") or ""
            message = message or errors[0].get("message", "") or ""
    except Exception:
        details = getattr(err, "error_details", None) or []
        if details and isinstance(details[0], dict):
            reason = details[0].get("reason", "") or ""
            message = details[0].get("message", "") or ""
    if not message:
        message = str(err)

    match = re.search(r"project_number:(\d+)", message)
    if match:
        project = match.group(1)
    return reason, message, project


def _is_daily_quota_error(reason: str, message: str, status) -> bool:
    """True when this error means the current key is out of quota for the day."""
    if reason in QUOTA_REASONS:
        return True
    low = message.lower().replace("-", " ")
    if any(marker in low for marker in DAILY_QUOTA_MARKERS):
        return True                      # 429 rateLimitExceeded that is really a daily cap
    return status == 403 and "quota" in low


class ApiKeyPool:
    """Round-robin over the YOUTUBE_API_KEY_* values, skipping exhausted keys."""

    def __init__(self):
        self.keys = []
        i = 1
        while (key := os.getenv(f"YOUTUBE_API_KEY_{i}")):
            self.keys.append((f"YOUTUBE_API_KEY_{i}", key.strip()))
            i += 1
        if not self.keys:
            raise ValueError("No YOUTUBE_API_KEY_* found — check your .env file")

        self.idx = 0
        self.exhausted = set()
        self.units_used = 0
        self._client = None

        # key name -> Google Cloud project number. A key's project is only revealed in its
        # quota-error message, so we cache what we learn and reuse it on later runs.
        self.projects = {}
        if KEY_PROJECTS_CACHE.exists():
            try:
                self.projects = json.loads(KEY_PROJECTS_CACHE.read_text())
            except Exception:
                self.projects = {}

        shared = {}
        for kname, proj in self.projects.items():
            shared.setdefault(proj, []).append(kname)
        for proj, names in shared.items():
            if len(names) > 1:
                print(f"⚠️  {', '.join(names)} are all in Cloud project {proj} — they SHARE one "
                      f"daily quota. Effective capacity is that of a single key.")

    @property
    def name(self) -> str:
        return self.keys[self.idx][0]

    @property
    def client(self):
        if self._client is None:
            self._client = build("youtube", "v3", developerKey=self.keys[self.idx][1],
                                 cache_discovery=False)
        return self._client

    def _note_project(self, project: str):
        """Record which Cloud project a key belongs to, and persist it for future runs."""
        if not project:
            return
        self.projects[self.name] = project
        try:
            KEY_PROJECTS_CACHE.write_text(json.dumps(self.projects, indent=2))
        except Exception:
            pass

    def mark_exhausted(self, project: str = ""):
        """Retire the current key and move to the next live one."""
        self._note_project(project)
        self.exhausted.add(self.idx)
        print(f"    ⚠️  {self.name} out of daily quota ({len(self.exhausted)}/{len(self.keys)} exhausted)")

        # Keys in an already-exhausted project are dead too — don't waste a call proving it.
        # (Only possible for keys whose project we already know, i.e. from the cache.)
        if project:
            for i, (kname, _) in enumerate(self.keys):
                if i not in self.exhausted and self.projects.get(kname) == project:
                    self.exhausted.add(i)
                    print(f"    ⚠️  {kname} is in the same exhausted project ({project}) — skipping it")

        for step in range(1, len(self.keys) + 1):
            nxt = (self.idx + step) % len(self.keys)
            if nxt not in self.exhausted:
                self.idx = nxt
                self._client = None
                print(f"    🔑 switching to {self.name}")
                return
        raise AllKeysExhausted(
            f"All {len(self.keys)} API key(s) have hit their daily quota. "
            f"Progress is saved — re-run this cell after the quota resets (midnight Pacific)."
        )

    def execute(self, make_request, cost: int, max_transient_retries: int = 4):
        """Run `make_request(youtube_client).execute()`, rotating keys / retrying as needed."""
        attempt = 0
        while True:
            try:
                result = make_request(self.client).execute()
                self.units_used += cost
                return result
            except HttpError as err:
                status = getattr(err.resp, "status", None)
                reason, message, project = _error_info(err)

                if _is_daily_quota_error(reason, message, status):
                    self.mark_exhausted(project)   # raises AllKeysExhausted when nothing is left
                    attempt = 0                    # fresh key, fresh retry budget
                    continue

                if reason in TRANSIENT_REASONS or (status is not None and status >= 500):
                    attempt += 1
                    if attempt > max_transient_retries:
                        raise
                    wait = 2 ** attempt
                    print(f"    ⏳ transient error ({reason or status}) — retrying in {wait}s")
                    time.sleep(wait)
                    continue

                raise


pool = ApiKeyPool()
n_projects = len(set(pool.projects.values())) or None

print(f"Loaded {len(pool.keys)} API key(s): {', '.join(n for n, _ in pool.keys)}")
if n_projects and len(pool.projects) == len(pool.keys):
    capacity = n_projects * 10_000 // (COST_SEARCH + COST_VIDEOS)
    print(f"Distinct Cloud projects: {n_projects} → ~{capacity} CEO/year pairs per day")
else:
    capacity = len(pool.keys) * 10_000 // (COST_SEARCH + COST_VIDEOS)
    print(f"Capacity today: up to ~{capacity} CEO/year pairs — but ONLY if each key is in a "
          f"different Google Cloud project.")
    print("Quota is charged per PROJECT, not per key: 4 keys in one project = 10,000 units total,")
    print("not 40,000. Each key's project is learned from its first quota error and cached in")
    print(f"{KEY_PROJECTS_CACHE}.")

Loaded 15 API key(s): YOUTUBE_API_KEY_1, YOUTUBE_API_KEY_2, YOUTUBE_API_KEY_3, YOUTUBE_API_KEY_4, YOUTUBE_API_KEY_5, YOUTUBE_API_KEY_6, YOUTUBE_API_KEY_7, YOUTUBE_API_KEY_8, YOUTUBE_API_KEY_9, YOUTUBE_API_KEY_10, YOUTUBE_API_KEY_11, YOUTUBE_API_KEY_12, YOUTUBE_API_KEY_13, YOUTUBE_API_KEY_14, YOUTUBE_API_KEY_15
Capacity today: up to ~1485 CEO/year pairs — but ONLY if each key is in a different Google Cloud project.
Quota is charged per PROJECT, not per key: 4 keys in one project = 10,000 units total,
not 40,000. Each key's project is learned from its first quota error and cached in
data/output/videos_metadata/key_projects.json.


## 3. Load the CEO list and any progress from previous runs

In [56]:
def pair_key(year, company) -> str:
    """Unique id for a CEO/year pair. (year, company) is unique in ceos.csv."""
    return f"{int(year)}|{company}"

def record_key(year, company, slot: str = "first") -> str:
    """
    Unique id for a record. Co-CEO companies produce two records for the same
    (year, company), so the second one is tagged to keep them apart.
    First-slot keys are unchanged, so records written before `second_CEO` existed
    keep working exactly as they did.
    """
    base = pair_key(year, company)
    return base if slot == "first" else f"{base}|{slot}"

def _norm_name(value) -> str | None:
    """Normalise a CEO name for comparison. Missing/blank -> None."""
    if value is None or (isinstance(value, float) and pd.isna(value)) or pd.isna(value):
        return None
    text = str(value).strip()
    return text or None

def load_progress() -> dict:
    """Read whatever has already been collected. The .jsonl checkpoint wins over the .json."""
    records = {}
    if OUTPUT_JSON.exists():
        with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
            for rec in json.load(f):
                records[record_key(rec["year"], rec["company"], rec.get("ceo_slot", "first"))] = rec
    if OUTPUT_JSONL.exists():
        with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rec = json.loads(line)
                    key = record_key(rec["year"], rec["company"], rec.get("ceo_slot", "first"))
                    records[key] = rec  # later lines win
    return records

def append_checkpoint(record: dict):
    """Flush one finished CEO/year pair to disk immediately (O(1), crash-safe)."""
    with open(OUTPUT_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def write_all_videos_json(records: dict):
    """(Re)build the deliverable all_videos.json from the in-memory records."""
    ordered = sorted(records.values(),
                     key=lambda r: (-int(r["year"]), r.get("rank") or 9999,
                                    r.get("ceo_slot", "first") != "first"))
    tmp = OUTPUT_JSON.with_suffix(".json.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(ordered, f, ensure_ascii=False, indent=2)
    tmp.replace(OUTPUT_JSON)   # atomic — never leaves a half-written file behind
    return OUTPUT_JSON.stat().st_size

ceos = pd.read_csv(CEOS_CSV)
for _col in ("CEO", "second_CEO"):
    ceos[_col] = ceos[_col].astype("string").str.strip()
    ceos.loc[ceos[_col].isin(["", "nan", "None"]), _col] = pd.NA

records = load_progress()

print(f"CEO/year pairs in ceos.csv : {len(ceos):,}")
print(f"  with a CEO name          : {ceos['CEO'].notna().sum():,}")
print(f"  missing a CEO name (NA)  : {ceos['CEO'].isna().sum():,}")
print(f"  with a second_CEO        : {ceos['second_CEO'].notna().sum():,}")
print(f"Already collected          : {len(records):,}")


CEO/year pairs in ceos.csv : 3,000
  with a CEO name          : 3,000
  missing a CEO name (NA)  : 0
  with a second_CEO        : 39
Already collected          : 3,039


## 4. Collect the raw video metadata

Re-run this cell as often as you like. It picks up exactly where it left off.

For each pair it does two calls:
1. `search.list` — up to 50 videos for `"[CEO] interview"` published inside that calendar year;
2. `videos.list` — the full `snippet` + `contentDetails` + `statistics` + `status` +
   `topicDetails` record for those video ids.

The stored `videos` entries carry the **complete** API payload, plus `search_rank`
(relevance position) and `video_id` at the top level for convenience.

### What gets (re)collected

`collection_status()` classifies every CEO/year pair, checking for a **changed CEO name first**:

| status | when | effect |
|---|---|---|
| **`stale`** | the `CEO` in `ceos.csv` ≠ the `CEO` stored in `all_videos.json` for that same year + company | **discard the saved videos and search again under the new name** — regardless of whether the pair had 50 videos, 0, or `"NA"` |
| `todo` | never collected | search |
| `retry` | same name, previously `NA` (and `RETRY_NA`) or `0` videos (and `RETRY_ZERO`) | search |
| `skip` | same name, already has videos | left untouched |

Names are compared **exactly**, after trimming surrounding whitespace; a blank/`NaN` name on
either side counts as "missing", so a still-empty CSV cell against a stored `"NA"` is *not* a
change. Correcting `Andy Jassy` → `Andrew Jassy` **is** a change and forces a fresh search.

A re-collected record keeps the name it replaced in a **`replaced_ceo`** field, so you can
always tell which rows were redone and what they used to hold.


In [57]:
def build_record(row, videos=None, n_found=None, note=None, previous_ceo=None) -> dict:
    """Assemble the JSON record for one CEO/year pair."""
    ceo = row["CEO"]
    has_ceo = pd.notna(ceo)
    return {
        "year": int(row["year"]),
        "company": row["company"],
        "rank": int(row["rank"]) if pd.notna(row["rank"]) else None,
        "CEO": ceo if has_ceo else None,
        "CEO_alt_names": row["CEO_alt_names"] if pd.notna(row["CEO_alt_names"]) else None,
        "query": QUERY_TEMPLATE.format(ceo=ceo) if has_ceo else None,
        "published_after": f"{int(row['year'])}-01-01T00:00:00Z" if has_ceo else None,
        "published_before": f"{int(row['year'])}-12-31T23:59:59Z" if has_ceo else None,
        # int when we could search, the string "NA" when the CEO name is missing
        "n_videos_found": n_found if n_found is not None else "NA",
        "collected_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        # set when this pair was re-collected because the CEO name changed in ceos.csv
        "replaced_ceo": previous_ceo,
        "note": note,
        "videos": videos or [],
    }


def search_ceo_year(ceo: str, year: int) -> list:
    """Search YouTube and return the full metadata of every video returned."""
    search = pool.execute(
        lambda yt: yt.search().list(
            part="snippet",
            q=QUERY_TEMPLATE.format(ceo=ceo),
            type="video",
            maxResults=MAX_RESULTS,
            order="relevance",
            publishedAfter=f"{year}-01-01T00:00:00Z",
            publishedBefore=f"{year}-12-31T23:59:59Z",
        ),
        cost=COST_SEARCH,
    )

    items = search.get("items", [])
    if not items:
        return []

    # Keep the relevance ordering from the search response.
    order = {it["id"]["videoId"]: i for i, it in enumerate(items) if it["id"].get("videoId")}
    search_snippets = {it["id"]["videoId"]: it["snippet"] for it in items if it["id"].get("videoId")}

    details = pool.execute(
        lambda yt: yt.videos().list(
            part="snippet,contentDetails,statistics,status,topicDetails",
            id=",".join(order.keys()),
            maxResults=MAX_RESULTS,
        ),
        cost=COST_VIDEOS,
    )
    detail_by_id = {v["id"]: v for v in details.get("items", [])}

    videos = []
    for vid, rank in order.items():
        full = detail_by_id.get(vid)
        if full is not None:
            entry = dict(full)                      # complete videos.list payload
        else:
            # Video vanished between the two calls (deleted/private) — keep the search data.
            entry = {"id": vid, "snippet": search_snippets[vid], "unavailable_in_videos_list": True}
        entry["video_id"] = vid
        entry["search_rank"] = rank
        videos.append(entry)

    videos.sort(key=lambda v: v["search_rank"])
    return videos


def collection_status(row) -> str:
    """
    Decide what to do with one CEO/year pair.

      "stale"   the CEO name in ceos.csv differs from the one stored in all_videos.json
                -> discard whatever was saved and search again under the new name
      "todo"    never collected before
      "retry"   previously NA / 0 videos, and the matching RETRY_* flag is on
      "skip"    already collected under the same name — leave untouched
    """
    existing = records.get(pair_key(row["year"], row["company"]))
    if existing is None:
        return "todo"

    # ── CEO name change wins over everything else ────────────────────────────
    # Compared exactly (after trimming whitespace): a different name means the stored
    # videos belong to the wrong person, whether there were 50 of them, 0, or "NA".
    if _norm_name(existing.get("CEO")) != _norm_name(row["CEO"]):
        return "stale"

    # ── Same name: the previous NA / 0-video rules apply ─────────────────────
    if existing["n_videos_found"] == "NA":
        return "retry" if (RETRY_NA and pd.notna(row["CEO"])) else "skip"
    if existing["n_videos_found"] == 0:
        return "retry" if RETRY_ZERO else "skip"
    return "skip"                                   # has videos under the current name


# ── Build the work list ──────────────────────────────────────────────────────
todo = ceos if YEARS_FILTER is None else ceos[ceos["year"].isin(YEARS_FILTER)]
todo = todo.sort_values(["year", "rank"], ascending=[False, True])

status = todo.apply(collection_status, axis=1)
stale_keys = {pair_key(r["year"], r["company"])
              for _, r in todo[status == "stale"].iterrows()}
todo = todo[status != "skip"]

n_searchable = int(todo["CEO"].notna().sum())
print(f"Pairs to process this run : {len(todo):,}  ({n_searchable:,} searches + "
      f"{len(todo) - n_searchable:,} NA rows)")
if stale_keys:
    print(f"  of which CEO name changed : {len(stale_keys):,}  "
          f"(old videos discarded and re-searched)")
print(f"Estimated quota needed    : {n_searchable * (COST_SEARCH + COST_VIDEOS):,} units "
      f"(available today: ~{len(pool.keys) * 10_000:,})\n")

# ── Main loop ────────────────────────────────────────────────────────────────
searches_done = 0
new_records = 0
stopped_reason = "finished"

try:
    for _, row in todo.iterrows():
        key = pair_key(row["year"], row["company"])

        was_stale = key in stale_keys
        previous_ceo = records[key].get("CEO") if was_stale else None

        if pd.isna(row["CEO"]):
            record = build_record(row, previous_ceo=previous_ceo,
                                  note="CEO name missing from ceos.csv — no search performed")
            records[key] = record
            append_checkpoint(record)
            new_records += 1
            continue

        if MAX_SEARCHES_THIS_RUN is not None and searches_done >= MAX_SEARCHES_THIS_RUN:
            stopped_reason = f"reached MAX_SEARCHES_THIS_RUN ({MAX_SEARCHES_THIS_RUN})"
            break

        try:
            videos = search_ceo_year(row["CEO"], int(row["year"]))
        except AllKeysExhausted as exc:
            stopped_reason = str(exc)
            break

        record = build_record(row, videos=videos, n_found=len(videos),
                              previous_ceo=previous_ceo)
        records[key] = record      # replaces the stale record outright
        append_checkpoint(record)          # progress saved after every CEO
        searches_done += 1
        new_records += 1

        changed = f"  (was: {previous_ceo})" if was_stale else ""
        print(f"[{searches_done:>4}] {row['year']}  {row['CEO']:<28.28} "
              f"{row['company']:<26.26} → {len(videos):>2} videos{changed}")

except KeyboardInterrupt:
    stopped_reason = "interrupted by user"

finally:
    size = write_all_videos_json(records)
    print(f"\n{'─' * 70}")
    print(f"Stopped: {stopped_reason}")
    print(f"New/updated records this run : {new_records:,}  ({searches_done:,} searches)")
    if stale_keys:
        print(f"Re-collected after name change: {len(stale_keys):,}")
    print(f"Quota units used this run    : {pool.units_used:,}")
    print(f"Keys exhausted               : {len(pool.exhausted)}/{len(pool.keys)}")
    print(f"Total records on disk        : {len(records):,}")
    print(f"Wrote {OUTPUT_JSON}  ({size / 1024**2:.1f} MB)")

Pairs to process this run : 141  (141 searches + 0 NA rows)
  of which CEO name changed : 141  (old videos discarded and re-searched)
Estimated quota needed    : 14,241 units (available today: ~150,000)

    ⚠️  YOUTUBE_API_KEY_1 out of daily quota (1/15 exhausted)
    🔑 switching to YOUTUBE_API_KEY_2
    ⚠️  YOUTUBE_API_KEY_2 out of daily quota (2/15 exhausted)
    🔑 switching to YOUTUBE_API_KEY_3
    ⚠️  YOUTUBE_API_KEY_3 out of daily quota (3/15 exhausted)
    🔑 switching to YOUTUBE_API_KEY_4
    ⚠️  YOUTUBE_API_KEY_4 out of daily quota (4/15 exhausted)
    🔑 switching to YOUTUBE_API_KEY_5
    ⚠️  YOUTUBE_API_KEY_5 out of daily quota (5/15 exhausted)
    🔑 switching to YOUTUBE_API_KEY_6
    ⚠️  YOUTUBE_API_KEY_6 out of daily quota (6/15 exhausted)
    🔑 switching to YOUTUBE_API_KEY_7
[   1] 2025  Michael K. Wirth             Chevron                    → 50 videos  (was: Mike Wirth)
[   2] 2025  Rainer M. Blair              Danaher                    →  8 videos  (was: Rainer Blair)


## 5. Summary statistics

Ignores the 500 stored **2026** records — that year was dropped from `ceos.csv`, so it is no
longer part of the project. The records stay on disk untouched; they are just excluded here.

Only needs cells 1 and 3 to have run (no API calls, no quota).


In [58]:
records = load_progress()

# ceos.csv no longer contains 2026, but all_videos.json still holds the 500 records
# collected for it. They are kept on disk and simply ignored everywhere below.
YEARS = sorted(ceos["year"].unique())
stats = [r for r in records.values() if r["year"] in YEARS]
ignored = len(records) - len(stats)

na    = [r for r in stats if r["n_videos_found"] == "NA"]
zero  = [r for r in stats if r["n_videos_found"] == 0]
found = [r for r in stats if r["n_videos_found"] not in ("NA", 0)]
searched = zero + found                          # pairs an actual search ran for

# ── CEO name mismatches between ceos.csv and all_videos.json ────────────────
# Same comparison the collection cell uses, so this is exactly the number of pairs
# that would be discarded and re-searched on the next run.
mismatches = [
    (int(row["year"]), row["company"], stored.get("CEO"), row["CEO"])
    for _, row in ceos.iterrows()
    if (stored := records.get(pair_key(row["year"], row["company"]))) is not None
    and _norm_name(stored.get("CEO")) != _norm_name(row["CEO"])
]

total_videos = sum(r["n_videos_found"] for r in searched)
pending = len(ceos) - len(stats)

print("=" * 62)
print(f"COLLECTION SUMMARY  ({YEARS[0]}–{YEARS[-1]})".center(62))
print("=" * 62)
print(f"CEO/year pairs in ceos.csv   : {len(ceos):>7,}")
print(f"  collected                  : {len(stats):>7,}  ({len(stats) / len(ceos):.1%})")
print(f"  still pending              : {pending:>7,}")
if ignored:
    print(f"(ignoring {ignored:,} stored 2026 records — no longer in ceos.csv)")

print()
print(f"NA — CEO name missing        : {len(na):>7,}")
print(f"0 videos found               : {len(zero):>7,}")
print(f"1+ videos found              : {len(found):>7,}")

print()
print(f"Total videos collected       : {total_videos:>7,}")
if searched:
    print(f"Average videos per pair      : {total_videos / len(searched):>10.1f}   "
          f"(over the {len(searched):,} pairs actually searched)")

print()
print(f"CEO name mismatches          : {len(mismatches):>7,}")
if mismatches:
    n_filled = sum(1 for *_, stored_ceo, _ in mismatches if _norm_name(stored_ceo) is None)
    print(f"  was NA, now named in csv   : {n_filled:>7,}")
    print(f"  renamed                    : {len(mismatches) - n_filled:>7,}")
    print("  → these are discarded and re-searched on the next run of cell 4")
    print()
    print("  examples:")
    for year, company, stored_ceo, csv_ceo in mismatches[:5]:
        print(f"    {year}  {company:<26.26} {str(stored_ceo):<22.22} → {csv_ceo}")


               COLLECTION SUMMARY  (2020–2025)                
CEO/year pairs in ceos.csv   :   3,000
  collected                  :   3,039  (101.3%)
  still pending              :     -39

NA — CEO name missing        :       0
0 videos found               :     167
1+ videos found              :   2,872

Total videos collected       : 104,641
Average videos per pair      :       34.4   (over the 3,039 pairs actually searched)

CEO name mismatches          :       0


## 6. Get videos for the second CEO

Some companies are run by **co-CEOs**, and `ceos.csv` carries the other one in
`second_CEO` (39 rows). This cell is cell 4 all over again, but reading **only** the
`second_CEO` column, and it writes into the same `all_videos.json` in the same record shape.

Rows with no `second_CEO` are skipped entirely — they are not co-CEO companies, so they get
no second record at all (not even an `"NA"` one; that is what cell 4's `NA` already means).

Because `(year, company)` no longer identifies a record on its own, second-CEO records carry
**`"ceo_slot": "second"`** and are keyed separately, so Netflix 2025 holds *two* records —
Ted Sarandos and Greg Peters — instead of one overwriting the other. Records written before
this existed have no `ceo_slot` and are read as `"first"`, so nothing needed migrating.

Same behaviour as cell 4 otherwise: resumable, skips what it already has, re-searches a pair
whose name changed in the CSV, saves after every CEO, and stops cleanly when quota runs out.
Needs cells 1, 2, 3 and the function definitions in cell 4 to have run first.


In [59]:
CEO_COLUMN  = "second_CEO"          # this cell reads ONLY this column
CEO_SLOT    = "second"              # tags the records it writes


def build_second_record(row, videos=None, n_found=None, note=None, previous_ceo=None) -> dict:
    """Same record shape as cell 4, but built from second_CEO."""
    ceo = row[CEO_COLUMN]
    has_ceo = pd.notna(ceo)
    return {
        "year": int(row["year"]),
        "company": row["company"],
        "rank": int(row["rank"]) if pd.notna(row["rank"]) else None,
        "CEO": ceo if has_ceo else None,
        "CEO_alt_names": row["second_CEO_alt_names"] if pd.notna(row["second_CEO_alt_names"]) else None,
        # marks this as the co-CEO record; the first-CEO record for the same
        # (year, company) is stored alongside it with ceo_slot == "first"
        "ceo_slot": CEO_SLOT,
        "query": QUERY_TEMPLATE.format(ceo=ceo) if has_ceo else None,
        "published_after": f"{int(row['year'])}-01-01T00:00:00Z" if has_ceo else None,
        "published_before": f"{int(row['year'])}-12-31T23:59:59Z" if has_ceo else None,
        # int when we could search, the string "NA" when the CEO name is missing
        "n_videos_found": n_found if n_found is not None else "NA",
        "collected_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        # set when this pair was re-collected because the CEO name changed in ceos.csv
        "replaced_ceo": previous_ceo,
        "note": note,
        "videos": videos or [],
    }


def second_collection_status(row) -> str:
    """Same rules as cell 4's collection_status, applied to the second_CEO column."""
    existing = records.get(record_key(row["year"], row["company"], CEO_SLOT))
    if existing is None:
        return "todo"

    # A changed name wins over everything else — discard and re-search.
    if _norm_name(existing.get("CEO")) != _norm_name(row[CEO_COLUMN]):
        return "stale"

    if existing["n_videos_found"] == "NA":
        return "retry" if (RETRY_NA and pd.notna(row[CEO_COLUMN])) else "skip"
    if existing["n_videos_found"] == 0:
        return "retry" if RETRY_ZERO else "skip"
    return "skip"


# ── Build the work list ──────────────────────────────────────────────────────
# Only rows that actually name a second CEO. Rows without one are not co-CEO
# companies at all, so they get no second record (not even an "NA" one).
todo2 = ceos if YEARS_FILTER is None else ceos[ceos["year"].isin(YEARS_FILTER)]
todo2 = todo2[todo2[CEO_COLUMN].notna()].sort_values(["year", "rank"], ascending=[False, True])

status2 = todo2.apply(second_collection_status, axis=1) if len(todo2) else pd.Series(dtype=object)
stale_keys2 = {record_key(r["year"], r["company"], CEO_SLOT)
               for _, r in todo2[status2 == "stale"].iterrows()} if len(todo2) else set()
todo2 = todo2[status2 != "skip"] if len(todo2) else todo2

print(f"Rows with a second CEO    : {int(ceos[CEO_COLUMN].notna().sum()):,}")
print(f"To process this run       : {len(todo2):,}")
if stale_keys2:
    print(f"  of which name changed   : {len(stale_keys2):,}  (old videos discarded and re-searched)")
print(f"Estimated quota needed    : {len(todo2) * (COST_SEARCH + COST_VIDEOS):,} units\n")

# ── Main loop ────────────────────────────────────────────────────────────────
searches_done2 = 0
new_records2 = 0
stopped_reason2 = "finished"

try:
    for _, row in todo2.iterrows():
        key = record_key(row["year"], row["company"], CEO_SLOT)

        was_stale = key in stale_keys2
        previous_ceo = records[key].get("CEO") if was_stale else None

        if MAX_SEARCHES_THIS_RUN is not None and searches_done2 >= MAX_SEARCHES_THIS_RUN:
            stopped_reason2 = f"reached MAX_SEARCHES_THIS_RUN ({MAX_SEARCHES_THIS_RUN})"
            break

        try:
            videos = search_ceo_year(row[CEO_COLUMN], int(row["year"]))
        except AllKeysExhausted as exc:
            stopped_reason2 = str(exc)
            break

        record = build_second_record(row, videos=videos, n_found=len(videos),
                                     previous_ceo=previous_ceo)
        records[key] = record              # never touches the first-CEO record
        append_checkpoint(record)          # progress saved after every CEO
        searches_done2 += 1
        new_records2 += 1

        changed = f"  (was: {previous_ceo})" if was_stale else ""
        print(f"[{searches_done2:>4}] {row['year']}  {row[CEO_COLUMN]:<28.28} "
              f"{row['company']:<26.26} → {len(videos):>2} videos{changed}")

except KeyboardInterrupt:
    stopped_reason2 = "interrupted by user"

finally:
    size = write_all_videos_json(records)
    print(f"\n{'─' * 70}")
    print(f"Stopped: {stopped_reason2}")
    print(f"New/updated second-CEO records : {new_records2:,}  ({searches_done2:,} searches)")
    if stale_keys2:
        print(f"Re-collected after name change : {len(stale_keys2):,}")
    print(f"Quota units used this run      : {pool.units_used:,}")
    print(f"Keys exhausted                 : {len(pool.exhausted)}/{len(pool.keys)}")
    print(f"Total records on disk          : {len(records):,}")
    print(f"Wrote {OUTPUT_JSON}  ({size / 1024**2:.1f} MB)")


Rows with a second CEO    : 39
To process this run       : 0
Estimated quota needed    : 0 units


──────────────────────────────────────────────────────────────────────
Stopped: finished
New/updated second-CEO records : 0  (0 searches)
Quota units used this run      : 14,237
Keys exhausted                 : 13/15
Total records on disk          : 3,039
Wrote data/output/videos_metadata/all_videos.json  (473.6 MB)


## 7. Total summary statistics — first + second CEOs

Everything cell 5 reports, but counting **both** CEO slots. A co-CEO company contributes
**two** pairs to the denominator, so the project expects `len(ceos)` + one per `second_CEO`
row — 3,039 pairs, not 3,000.

"Still missing" counts pairs that have no record at all yet, split by slot, so an
un-run cell 6 shows up as 39 missing second CEOs rather than silently reading 100% done.
Mismatches are checked in both slots and tagged `[first]` / `[second]`.

Records outside the years in `ceos.csv` (the archived 2026 set) are ignored.
Needs cells 1 and 3 to have run — no API calls, no quota.


In [60]:
records = load_progress()

# 2026 was dropped from ceos.csv; any records still stored for it are ignored here.
YEARS = sorted(ceos["year"].unique())
stats = [r for r in records.values() if r["year"] in YEARS]
ignored = len(records) - len(stats)

# ── Every CEO/year pair the project expects ─────────────────────────────────
# One per CSV row for the first CEO, plus one more for each row naming a co-CEO.
expected = [(int(r["year"]), r["company"], "first", r["CEO"]) for _, r in ceos.iterrows()]
expected += [(int(r["year"]), r["company"], "second", r["second_CEO"])
             for _, r in ceos[ceos["second_CEO"].notna()].iterrows()]

by_slot = {"first": [], "second": []}
for r in stats:
    by_slot[r.get("ceo_slot", "first")].append(r)

na    = [r for r in stats if r["n_videos_found"] == "NA"]
zero  = [r for r in stats if r["n_videos_found"] == 0]
found = [r for r in stats if r["n_videos_found"] not in ("NA", 0)]
searched = zero + found                      # pairs an actual search ran for
total_videos = sum(r["n_videos_found"] for r in searched)

# ── Pairs expected but not yet in the file ──────────────────────────────────
missing = [(y, c, slot, name) for y, c, slot, name in expected
           if record_key(y, c, slot) not in records]

# ── Name mismatches, checked in both slots ──────────────────────────────────
# Same comparison the collection cells use, so this is exactly what would be
# discarded and re-searched on the next run.
mismatches = [
    (y, c, slot, stored.get("CEO"), name)
    for y, c, slot, name in expected
    if (stored := records.get(record_key(y, c, slot))) is not None
    and _norm_name(stored.get("CEO")) != _norm_name(name)
]

print("=" * 64)
print(f"TOTAL SUMMARY — first + second CEOs  ({YEARS[0]}–{YEARS[-1]})".center(64))
print("=" * 64)
print(f"Unique CEO/year pairs expected : {len(expected):>7,}")
print(f"  first CEO  (one per csv row) : {len(ceos):>7,}")
print(f"  second CEO (co-CEO rows)     : {int(ceos['second_CEO'].notna().sum()):>7,}")

print()
print(f"Collected                      : {len(stats):>7,}  ({len(stats) / len(expected):.1%})")
print(f"  first CEO records            : {len(by_slot['first']):>7,}")
print(f"  second CEO records           : {len(by_slot['second']):>7,}")
print(f"Still missing                  : {len(missing):>7,}")
if missing:
    n_first = sum(1 for _, _, slot, _ in missing if slot == "first")
    print(f"  first CEO                    : {n_first:>7,}")
    print(f"  second CEO                   : {len(missing) - n_first:>7,}")
if ignored:
    print(f"(ignoring {ignored:,} stored records outside {YEARS[0]}–{YEARS[-1]})")

print()
print(f"NA — CEO name missing          : {len(na):>7,}")
print(f"0 videos found                 : {len(zero):>7,}")
print(f"1+ videos found                : {len(found):>7,}")

print()
print(f"Total videos collected         : {total_videos:>7,}")
if searched:
    print(f"Average videos per pair        : {total_videos / len(searched):>10.1f}   "
          f"(over the {len(searched):,} pairs actually searched)")
if by_slot["second"]:
    sec_searched = [r for r in by_slot["second"] if r["n_videos_found"] != "NA"]
    sec_videos = sum(r["n_videos_found"] for r in sec_searched)
    print(f"  from second CEOs             : {sec_videos:>7,}"
          + (f"   (avg {sec_videos / len(sec_searched):.1f})" if sec_searched else ""))

print()
print(f"CEO name mismatches            : {len(mismatches):>7,}")
if mismatches:
    n_filled = sum(1 for *_, stored_ceo, _ in mismatches if _norm_name(stored_ceo) is None)
    print(f"  was NA, now named in csv     : {n_filled:>7,}")
    print(f"  renamed                      : {len(mismatches) - n_filled:>7,}")
    print("  → re-searched on the next run of cell 4 (first) / cell 6 (second)")
    print()
    print("  examples:")
    for year, company, slot, stored_ceo, csv_ceo in mismatches[:5]:
        print(f"    {year}  {company:<24.24} [{slot:<6}] {str(stored_ceo):<20.20} → {csv_ceo}")

remaining_searches = len(missing) + len(mismatches)
if remaining_searches:
    print()
    print(f"Searches still required        : {remaining_searches:>7,}  "
          f"({remaining_searches * (COST_SEARCH + COST_VIDEOS):,} quota units)")


        TOTAL SUMMARY — first + second CEOs  (2020–2025)        
Unique CEO/year pairs expected :   3,039
  first CEO  (one per csv row) :   3,000
  second CEO (co-CEO rows)     :      39

Collected                      :   3,039  (100.0%)
  first CEO records            :   3,000
  second CEO records           :      39
Still missing                  :       0

NA — CEO name missing          :       0
0 videos found                 :     167
1+ videos found                :   2,872

Total videos collected         : 104,641
Average videos per pair        :       34.4   (over the 3,039 pairs actually searched)
  from second CEOs             :   1,198   (avg 30.7)

CEO name mismatches            :       0


## Simple statistics

In [61]:
import pandas as pd

df = pd.read_csv("data/ceos.csv")

ceos = pd.concat([
    df[["year", "company", "CEO"]].rename(columns={"CEO": "ceo"}),
    df[["year", "company", "second_CEO"]].rename(columns={"second_CEO": "ceo"}),
]).dropna(subset=["ceo"])

ceos["ceo"] = ceos["ceo"].astype(str).str.strip()
ceos = ceos[ceos["ceo"] != ""]

print("Unique CEOs:", ceos["ceo"].nunique())
print("Total CEO/year pairs:", len(ceos))
print("Unique CEO/company pairs:", ceos[["ceo", "company"]].drop_duplicates().shape[0])

avg_ceos_per_company = (
    ceos[["company", "ceo"]]
    .drop_duplicates()
    .groupby("company")["ceo"]
    .nunique()
    .mean()
)

print("Avg CEOs per company:", avg_ceos_per_company)

Unique CEOs: 920
Total CEO/year pairs: 3039
Unique CEO/company pairs: 930
Avg CEOs per company: 1.5296052631578947
